<div style="background: linear-gradient(135deg, #0f2027, #203a43, #2c5364); padding: 28px; border-radius: 24px; text-align: center; color: #f2f5f7; box-shadow: 0 8px 20px rgba(0,0,0,0.12);">
  <h1 style="font-size: 42px; margin-bottom: 8px;">✨ Paired Generation ✨</h1>
  <h2 style="font-size: 24px; margin-top: 0;">02 · Part B — AE / VAE / cGAN from Scratch</h2>
  <p style="font-size: 18px;">Trained on LoLI-Street pairs (6000-image prototype at 256px). No pretrained weights.</p>
</div>

### Imports

In [ ]:
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import torch
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
from paired.autoencoder import ConvAE
from paired.datasets import PairedLoLI
from paired.pix2pix import UNetGenerator
from paired.vae import ConvVAE
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

<div style="background: linear-gradient(135deg, #74c69d, #48cae4); padding: 26px; border-radius: 22px; margin-top: 28px; border: 4px solid #2d6a4f; box-shadow: 0 8px 22px rgba(0, 0, 0, 0.18); color: #081c15;">
  <h1 style="color: #081c15; background-color: rgba(255,255,255,0.65); padding: 12px 16px; border-radius: 14px; margin-top: 0;">🧱 B.1 Convolutional AE (L1)</h1>
  <p style="font-size: 17px; line-height: 1.7; color: #133f30; font-weight: 500;">
    15 epochs, 6000 pairs: train L1 0.032, val L1 0.033. Full-val: PSNR 22.90, SSIM 0.786 —
    yet zero-shot mAP collapses to 0.271 (raw: 0.712). Pixel-fidelity ≠ task utility.
  </p>
</div>

### AE sample grid

In [ ]:
ae = ConvAE().to(dev)
ae.load_state_dict(torch.load(REPO / "checkpoints" / "ae.pt", map_location=dev, weights_only=True))
ae.eval()
ds = PairedLoLI("val", 256, subset=4)
fig, axes = plt.subplots(4, 3, figsize=(9, 12))
with torch.no_grad():
    for i, ax in enumerate(axes):
        low, high = ds[i]
        rec = ae(low.unsqueeze(0).to(dev))[0].cpu().permute(1, 2, 0).numpy()
        for a, t, im in zip(ax, ["low", "AE", "high"], [low.permute(1,2,0).numpy(), rec, high.permute(1,2,0).numpy()]):
            a.imshow(im); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

<div style="background: linear-gradient(135deg, #e8b7e8, #ffb6f4); padding: 24px; border-radius: 22px; margin-top: 28px; border: 4px solid #481344; color: #2b0a2a;">
  <h1 style="color: #2b0a2a;">🌫️ B.2 VAE (L1 + KL)</h1>
  <p style="font-size: 17px; line-height: 1.7; color: #2b0a2a;">
    Same skeleton + reparameterization + KL (beta=1e-3). Full-val: PSNR 23.07, SSIM 0.783,
    mAP 0.271 — indistinguishable from the AE on detection. The extra smoothness buys
    nothing for the detector; it may cost small objects (master-table Q1).
  </p>
</div>

<div style="background: linear-gradient(135deg, #f093fb, #f5576c); padding: 26px; border-radius: 22px; margin-top: 28px; border: 4px solid #a01040; box-shadow: 0 8px 22px rgba(0, 0, 0, 0.18); color: #fff0f5;">
  <h1 style="color: #fff0f5; background-color: rgba(255,255,255,0.15); padding: 12px 16px; border-radius: 14px; margin-top: 0;">⚔️ B.3 pix2pix cGAN</h1>
  <p style="font-size: 17px; line-height: 1.7; color: #ffe0e8;">
    U-Net + PatchGAN, adv + 100xL1. Prototype (20 epochs, 6000 pairs): PSNR 15.1, SSIM 0.63,
    mAP 0.127 — outputs are washed out (contrast std 31 vs 73 real). Undertrained, not broken:
    final pass (full 30k pairs, 25 more epochs from prototype weights) is training now.
    Target: PSNR &gt; 23, SSIM &gt; 0.6.
  </p>
</div>